In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd

import os
import json
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

from scipy.io.wavfile import write


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

/home/eternal/miniconda3/envs/vits_cp/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=4)
           2	LOAD_FAST(arg=0, lineno=7)
           4	LOAD_ATTR(arg=0, lineno=7)
           6	LOAD_CONST(arg=1, lineno=7)
           8	BINARY_SUBSCR(arg=None, lineno=7)
          10	STORE_FAST(arg=4, lineno=7)
          12	LOAD_CONST(arg=2, lineno=8)
          14	STORE_FAST(arg=5, lineno=8)
          16	LOAD_GLOBAL(arg=1, lineno=9)
          18	LOAD_GLOBAL(arg=2, lineno=9)
          20	LOAD_FAST(arg=4, lineno=9)
          22	CALL_FUNCTION(arg=1, lineno=9)
          24	CALL_FUNCTION(arg=1, lineno=9)
          26	GET_ITER(arg=None, lineno=9)
>         28	FOR_ITER(arg=152, lineno=9)
          30	STORE_FAST(arg=6, lineno=9)
          32	LOAD_FAST(arg=0, lineno=10)
          34	LOAD_FAST(arg=6, lineno=10)
          36	BINARY_SUBSCR(arg=None, lineno=10)
          38	STORE_FAST(arg=7, lineno=10)
          40	LOAD_FAST(arg=1, lineno=11)
          42	LOAD_FAST(arg=6, lineno=11)
          44	BINARY_SUBSCR(arg=None, line

In [3]:
!ls checkpoints

/bin/bash: /home/eternal/miniconda3/envs/vits_cp/lib/libtinfo.so.6: no version information available (required by /bin/bash)
D_558000.pth
D_559000.pth
G_558000.pth
G_559000.pth
checkpoints
config.json
eval
events.out.tfevents.1768550421.DESKTOP-UMQATDH.1637.0
events.out.tfevents.1768550603.DESKTOP-UMQATDH.1741.0
events.out.tfevents.1768653734.DESKTOP-UMQATDH.2271.0
events.out.tfevents.1768653754.DESKTOP-UMQATDH.2437.0
events.out.tfevents.1768654837.DESKTOP-UMQATDH.2695.0
events.out.tfevents.1768724439.DESKTOP-UMQATDH.1021.0
events.out.tfevents.1768809817.DESKTOP-UMQATDH.539.0
events.out.tfevents.1768833515.DESKTOP-UMQATDH.2063.0
events.out.tfevents.1768894729.DESKTOP-UMQATDH.470.0
events.out.tfevents.1768980098.DESKTOP-UMQATDH.519.0
events.out.tfevents.1769075810.DESKTOP-UMQATDH.526.0
events.out.tfevents.1769152555.DESKTOP-UMQATDH.500.0
events.out.tfevents.1769239685.DESKTOP-UMQATDH.504.0
events.out.tfevents.1769352778.DESKTOP-UMQATDH.432.0
events.out.tfevents.1769415470.DESKTOP-UMQATD

## Single Speaker

In [4]:
hps = utils.get_hparams_from_file("configs/config-single-speaker.json")

In [5]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("checkpoints/G_559000.pth", net_g, None)

/home/eternal/miniconda3/envs/vits_cp/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


INFO:root:Loaded checkpoint 'checkpoints/G_559000.pth' (iteration 1036)


In [6]:
stn_tst = get_text("新春シャンソンショーで、生麦生米生卵を交互に食べながら、右目右耳右耳右目と12,345回唱えた結果、隣の客はよく柿食う客だと判명(はんめい)し、東京都特許許可局(とうきょうととっきょきょかきょく)의 0.1mg(ミリグラム) 単位の 複雑な 手続きが 滞った。",hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

# Arange

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
#import IPython.display as ipd

import os
import json
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

from scipy.io.wavfile import write



/home/eternal/miniconda3/envs/vits_cp/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


DEBUG:numba.core.byteflow:bytecode dump:
>          0	NOP(arg=None, lineno=4)
           2	LOAD_FAST(arg=0, lineno=7)
           4	LOAD_ATTR(arg=0, lineno=7)
           6	LOAD_CONST(arg=1, lineno=7)
           8	BINARY_SUBSCR(arg=None, lineno=7)
          10	STORE_FAST(arg=4, lineno=7)
          12	LOAD_CONST(arg=2, lineno=8)
          14	STORE_FAST(arg=5, lineno=8)
          16	LOAD_GLOBAL(arg=1, lineno=9)
          18	LOAD_GLOBAL(arg=2, lineno=9)
          20	LOAD_FAST(arg=4, lineno=9)
          22	CALL_FUNCTION(arg=1, lineno=9)
          24	CALL_FUNCTION(arg=1, lineno=9)
          26	GET_ITER(arg=None, lineno=9)
>         28	FOR_ITER(arg=152, lineno=9)
          30	STORE_FAST(arg=6, lineno=9)
          32	LOAD_FAST(arg=0, lineno=10)
          34	LOAD_FAST(arg=6, lineno=10)
          36	BINARY_SUBSCR(arg=None, lineno=10)
          38	STORE_FAST(arg=7, lineno=10)
          40	LOAD_FAST(arg=1, lineno=11)
          42	LOAD_FAST(arg=6, lineno=11)
          44	BINARY_SUBSCR(arg=None, line

In [2]:
import numpy as np
import time

In [3]:
def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

In [4]:
hps = utils.get_hparams_from_file("configs/config-single-speaker.json")

In [5]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model).cpu()
_ = net_g.eval()

_ = utils.load_checkpoint("checkpoints/G_559000.pth", net_g, None)

/home/eternal/miniconda3/envs/vits_cp/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


INFO:root:Loaded checkpoint 'checkpoints/G_559000.pth' (iteration 1036)


In [6]:
test_sentence=["新設の神社の新進気鋭の新米神官が、信者向けの神聖な神饌として、新鮮な生米から精성込めて炊き上げた熱々の生粥を、不器用な手つきで突きつつ、右目右耳右耳右目と呟きながら、隣の客はよく柿食う客だという噂を背景に、李も桃も桃も李も、桃も李も桃のうちという伝統的な言葉の調子を整えるため、竹垣に竹立てかけたのは竹立てかけたかったから竹立てかけたという複雑な言い訳を延々と繰り返しつつ、結局は赤巻紙青巻紙黄巻紙を素早く巻き取るという、支離滅裂かつ極めて難解な言語的修行に、たった一人で黙々と挑み続けている。",
              "現代社会における急速な技術革新は、私たちの生活を驚くほど豊かにし、かつてない利便性をもたらしましたが、その一方で、地球環境に与える負の影響も決して無視できない深刻なレベルに達しており、私たちは今、単なる目先の効率性や快適さを追求するだけでなく、次世代のために持続可能な発展を真剣に模索し、具体的な行動指針を策定することで、自然と最先端技術が調和し共存できる新しい時代のパラダイムを早急に構築しなければならない極めて重要な局面に立たされているのです。",
              "見知らぬ土地へと赴く旅というものは、単に日常の喧騒から一時的に逃れるための行為という枠を超えて、普段の生活の中では決して気づくことのできなかった本当の自分自身の内面と静かに向き合い、異文化の中で生きる他者の多様な価値観を深く理解し、道端に咲く名もなき花や初めて目にする壮大な景色の中に潜む些細な感動を積み重ねることで、最終的に自分の長い人生を精神的に支えてくれるかけがえのない記憶の断片を作り上げるという点において、私たちの豊かであるべき人生には欠かすことのできない極めて重要な要素であると断言できるでしょう。",
              "人工知能技術が飛躍的な進化を遂げたことにより、これまで人間が担ってきた知的労働の大部分が機械によって効率的に代替されてしまうのではないかという漠然とした不安や懸念が各所で提起されているのは事実ですが、人間特有の深い共感能力や自由な発想に基づく独創的な直感、そして非常に複雑な社会的背景を考慮した倫理的判断が求められる高度な領域においては、依然として人間の果たすべき役割は絶対的かつ代替不可能であるため、私たちは技術を単なる脅威ではなく強力な道具として賢明に活用し、人間本来の尊い価値をさらに高めていけるような進歩の方向性を真摯に模索し続けなければなりません。",
              "たった一冊の本を深く丁寧に読み解くという行為は、著者がその長い生涯をかけて心血を注ぎ獲得した貴重な知恵や経験を、わずか数時間という短い時間の中で自分自身の血肉として受け入れる壮大な知的探検であり、行間に隠された真意を鋭く読み解きながら自分自身の既存の考えと対照させる内省的なプロセスを経ることで、思考の幅が劇的に広がり、世界を多角的に捉える観点がより洗練されていくという素晴らしい経験ができるため、読書は時代が移り変わっても決して色褪せることのない最も価値のある学習方法の一つとして重宝され続けているのです。",
              "芸術というものは、目には見えない人間の複雑な内面世界を視覚的あるいは聴覚的な具体的な形態へと見事に具現化し、それを他者へと鮮明に伝える強力な媒体としての役割を果たしており、深い悲しみに沈んでいる人々には温かい癒しを届け、退屈な日常に疲弊している人々には斬新なインスピレーションを吹き込み、時には言葉では到底表現しきれないほど入り組んだ感情を浄化させてくれるという極めて重要な機能を担うことで、長い人類の歴史をより豊かに、そして美しく彩ってきた唯一無二の宝物のような存在であると言っても過言ではありません。",
              "私たちの先祖が長い年月をかけて遺してくれた貴重な伝統文化は、一つの民族としてのアイデンティティを形成する揺るぎない根幹となるものであるため、これを決して疎かに扱うことは許されず、単に過去の遺産を当時の原型のまま固定的に保存し続けることだけに固執するのではなく、現代的な感性を柔軟に取り入れることで新しい価値を創造し、若い世代の人々も抵抗感なく日常の中で楽しむことのできる「生きている文化」として次世代へ正しく継承していく不断の努力が、今まさに切実に求められている時期に来ているのです。",
              "真の意味でのコミュニケーションとは、単に言葉を機械的に交わすだけの形式的な対話の枠組みを超えて、相手の置かれた立場や心情を深く推察する「易地思之」の謙虚な姿勢で心の声に真摯に耳を傾けるプロセスであり、異なる社会的背景を持つ人々が揺るぎない信頼関係を基盤として強い連帯感を築き上げていく過程で発生する相乗効果は、個人の精神的な成長を促すことはもちろん、社会全体に蔓延する不必要な葛藤を解消し、真の調和と平和を導き出すために決定的な役割を果たすことになるでしょう。",
              "一度過ぎ去ってしまえば二度と取り戻すことのできない時間は、この世のどのような高価な財宝とも決して交換することのできない最も公平でありながらも時に残酷な資源であるため、私たちは毎日の生活の中で直面する些細な日常の断片を大切に慈しみ、「現在」という名の素晴らしい贈り物の中で全力を尽くして生きることで、将来自分自身の歩んできた足跡を静かに振り返った際に、一点の後悔も残らないような意味のある充実した日々を丁寧に積み重ねていかなければならないという普遍的な事実を、決して忘れてはならないのです。",
              "子供の頃から身の回りで起こる自然現象に対して絶えず素朴な疑問を抱き、自らの手でその正解を探求しようとする科学的な探究心を育むことは、単に教科書的な知識を丸暗記することよりも遥かに価値のある教育のあり方であり、このような旺盛な好奇心は将来、未開の領域に果敢に挑戦するための勇気の源泉となり、人類が直面している極めて複雑な諸問題を解決できる革新的なアイデアを創出するための強力な原動力になるという点において、基礎科学教育の重要性はどれほど強調してもしすぎるということは決してありません。",]

In [7]:
for _ in range(3):
    result=[]
    for text in test_sentence:
        start=time.perf_counter()
        stn_tst = get_text(text,hps)
        with torch.no_grad():
            x_tst = stn_tst.cpu().unsqueeze(0)
            x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cpu()
            audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
        end=time.perf_counter()
        
        elapsed=(end-start)*1000
        result.append({
            "text":text,
            "length":len(text),
            "time_ms":elapsed
        })
        print(f"{text[:15]}... → {elapsed:.2f}ms")
        

新設の神社の新進気鋭の新米神官... → 6632.48ms
現代社会における急速な技術革新... → 5782.55ms
見知らぬ土地へと赴く旅というも... → 6070.78ms
人工知能技術が飛躍的な進化を遂... → 6873.75ms
たった一冊の本を深く丁寧に読み... → 6028.04ms
芸術というものは、目には見えな... → 5785.34ms
私たちの先祖が長い年月をかけて... → 5757.49ms
真の意味でのコミュニケーション... → 5676.12ms
一度過ぎ去ってしまえば二度と取... → 5638.87ms
子供の頃から身の回りで起こる自... → 5910.82ms
新設の神社の新進気鋭の新米神官... → 6427.26ms
現代社会における急速な技術革新... → 5591.19ms
見知らぬ土地へと赴く旅というも... → 6003.04ms
人工知能技術が飛躍的な進化を遂... → 6949.44ms
たった一冊の本を深く丁寧に読み... → 6020.18ms
芸術というものは、目には見えな... → 5907.57ms
私たちの先祖が長い年月をかけて... → 5520.83ms
真の意味でのコミュニケーション... → 5549.44ms
一度過ぎ去ってしまえば二度と取... → 5575.86ms
子供の頃から身の回りで起こる自... → 5839.98ms
新設の神社の新進気鋭の新米神官... → 6620.20ms
現代社会における急速な技術革新... → 5607.95ms
見知らぬ土地へと赴く旅というも... → 6019.47ms
人工知能技術が飛躍的な進化を遂... → 6929.70ms
たった一冊の本を深く丁寧に読み... → 6019.47ms
芸術というものは、目には見えな... → 6138.05ms
私たちの先祖が長い年月をかけて... → 5553.11ms
真の意味でのコミュニケーション... → 5625.20ms
一度過ぎ去ってしまえば二度と取... → 5569.82ms
子供の頃から身の回りで起こる自... → 5806.38ms


In [10]:
times=[r["time_ms"]for r in result]
print(f"mean: {np.mean(times)}")
print(f"standard_deviation: {np.std(times)}")
print(f"min_time: {np.min(times):.2f}ms")
print(f"max_time: {np.max(times):.2f}ms")

mean: 5988.9363240999955
standard_deviation: 445.42278692203234
min_time: 5553.11ms
max_time: 6929.70ms


In [11]:
output = {
    "result": result,
    "stats": {
        "mean_ms": np.mean(times),
        "std_ms": np.std(times),
        "min_ms": np.min(times),
        "max_ms": np.max(times)
    }
}
with open("baseline_inference.json", "w", encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

## Multiple Speakers

In [ ]:
hps = utils.get_hparams_from_file("./configs/config-single-speaker.json")

In [ ]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    n_speakers=hps.data.n_speakers,
    **hps.model).cuda()
_ = net_g.eval()

_ = utils.load_checkpoint("/path/to/model.pth", net_g, None)

In [ ]:
stn_tst = get_text("こんにちは", hps)
with torch.no_grad():
    x_tst = stn_tst.cuda().unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)]).cuda()
    sid = torch.LongTensor([4]).cuda()
    audio = net_g.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.cpu().float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate, normalize=False))

### Voice Conversion

In [ ]:
dataset = TextAudioSpeakerLoader(hps.data.validation_files, hps.data)
collate_fn = TextAudioSpeakerCollate()
loader = DataLoader(dataset, num_workers=8, shuffle=False,
    batch_size=1, pin_memory=True,
    drop_last=True, collate_fn=collate_fn)
data_list = list(loader)

In [ ]:
with torch.no_grad():
    x, x_lengths, spec, spec_lengths, y, y_lengths, sid_src = [x.cuda() for x in data_list[0]]
    sid_tgt1 = torch.LongTensor([1]).cuda()
    sid_tgt2 = torch.LongTensor([2]).cuda()
    sid_tgt3 = torch.LongTensor([4]).cuda()
    audio1 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.cpu().float().numpy()
    audio2 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.cpu().float().numpy()
    audio3 = net_g.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.cpu().float().numpy()
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].cpu().numpy(), rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps.data.sampling_rate, normalize=False))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps.data.sampling_rate, normalize=False))